# Trabajo Ev 1

## Leer DataSet


In [1]:
import pandas as pd
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

raw_df = pd.read_csv("retail_store_sales.csv")
#respaldo y copia del dataset original

df = raw_df
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


##Limpieza de | Price Per Unit | Quantity | Total Spent (Busqueda de Nulos)

In [2]:
#comprobamos la cantidad de campos nulos
df.isna().sum()

,0
Transaction ID,0
Customer ID,0
Category,0
Item,1213
Price Per Unit,609
Quantity,604
Total Spent,604
Payment Method,0
Location,0
Transaction Date,0


Podemos concluir que  TODOS los registros tienen un "TRANSACTION_ID", "CUSTOMER_ID", "CATEGORY","PAYMENT METHOD",
"LOCATION" y "TRANSACTION DATE".

In [3]:
#comprobamos si existen transacciones repetidas
id_repetidos= raw_df['Transaction ID'].duplicated().sum()
#comprobamos si en general existen  registros repetidos (aunque si no hay transacciones repetidas esto se puede evitar)
datos_Repetidos = raw_df.duplicated().sum()

print(f"la cantidad de transacciones repetidas es: {id_repetidos}")
print(f"la cantidad de datos repetidos es: {datos_Repetidos}")

la cantidad de transacciones repetidas es: 0
la cantidad de datos repetidos es: 0


no existen errores de duplicación y cada transacción es unica

In [4]:
# En los siguientes 4 codigos se buscarán patrones en los que no se pueden recuperar los valores de las columnas "Price Per Unit"	"Quantity"	"Total Spent"
# Esto debido a que por ejemplo , no podemos sacar el total sin la cantidad ni el precio unitario
# En general si hay menos de 2 campos con datos en estas 3 columnas, el dato es irrecuperable.
# Busqueda para saber si "Precio", "Cantidad", "Total" son Nulos de manera simultanea

#Casos donde 'Price per unit', 'Quantity' y 'Total spent' son nulos
filtro_null_1 = df.loc[
      df['Price Per Unit'].isnull() & df['Quantity'].isnull() & df['Total Spent'].isnull(),
      ['Price Per Unit', 'Quantity', 'Total Spent']
    ]
filtro_null_1.shape

(0, 3)

In [5]:
#Casos donde 'Cantidad' y 'Total' son Nulos
filtro_null_2 = df.loc[
      df['Quantity'].isnull() & df['Total Spent'].isnull(),
      ['Price Per Unit', 'Quantity', 'Total Spent']
    ]
filtro_null_2.shape

(604, 3)

In [6]:
#Casos donde 'Price per Unit y 'Quantity' son Nulos
filtro_null_3 = df.loc[
      df['Price Per Unit'].isnull() & df['Quantity'].isnull(),
      ['Price Per Unit', 'Quantity', 'Total Spent']
    ]
filtro_null_3.shape

(0, 3)

In [7]:

#Casos donde 'Price per Unit y 'Total' son Nulos
filtro_null_4 =df.loc[
      df['Price Per Unit'].isnull() & df['Total Spent'].isnull(),
      ['Price Per Unit', 'Quantity', 'Total Spent']
    ]
filtro_null_4.shape

(0, 3)

Podemos observar que de los 4 escenarios posibles donde NO se pueden recuperar estos 3 datos, solo hay  604 escenarios donde no hay cantidad, ni precio unitario, en el resto de casos es recuperable la información por ende, solo limpiamos ese escenario.

In [8]:
#borramos los escenarios de 'Price per unit', 'Quantity' y 'Total spent' donde no es recuperable
df.drop(filtro_null_2.index, axis=0, inplace=True)
#revisamos los nulls denuevo
df.isna().sum()

,0
Transaction ID,0
Customer ID,0
Category,0
Item,609
Price Per Unit,609
Quantity,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


##Recuperando campos de | Price Per Unit | Quantity | Total Spent (Restaurando nulos)

Podemos observar del bloque de codigo anterior que los unicos campos nulos actuales entre 'Price Per Unit', 'Quantity' y 'Total Spent' solo hay 609 casos nulos de Price Per unit


In [9]:

#Buscaremos reparar los campos vacios de 'Price Per unit', Tomando el Total y partiendolo por la cantidad

#PRICE PER UNIT NULL

df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Total Spent'] / df['Quantity'])

df.isna().sum()



,0
Transaction ID,0
Customer ID,0
Category,0
Item,609
Price Per Unit,0
Quantity,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


## Restauracion de |Items|

antes de intentar restaurar los Items, necesitamos explorar información que podria tener relevancia con items como su categoria y valores unicos




In [13]:
#Contamos la cantidad de categorias que existen
cat_size = df['Category'].value_counts()
print(cat_size)

Category
Furniture                             1525
Electric household essentials         1516
Milk Products                         1513
Food                                  1507
Butchers                              1496
Beverages                             1496
Computers and electric accessories    1477
Patisserie                            1441
Name: count, dtype: int64


In [14]:
#items unicos por cada categoria
cat_unique = df.groupby('Category')['Item'].nunique()
print(cat_unique)

Category
Beverages                             25
Butchers                              25
Computers and electric accessories    25
Electric household essentials         25
Food                                  25
Furniture                             25
Milk Products                         25
Patisserie                            25
Name: Item, dtype: int64


In [15]:
#items unicos
items_unique = df.loc[df['Item'].notna(), 'Item'].unique()
print(len(items_unique))

200


hay 200 items unicos y 8 categorias, por cada categoria hay 25 items unicos que no se repite.

Ahora vereficaremos si cada item tiene un unico valor unitario.

In [16]:
#verificaremos que cada item tenga un valor unitario unico (es de cir que un mismo item no tenga 2 valores unitario distintos)

#eliminamos los campos nulos de item para ahorrarnos problemas con el groupby y lo guardaremos en un DF temporal 'not_null_items'
not_null_items = df.loc[df['Item'].notna()].copy()

#hacemos un group by de items donde cada item guardara un array de todos los 'Price Per Unit' unicos que hay
item_prices_array = not_null_items.groupby('Item')['Price Per Unit'].unique()

#evaluamos en 'resultado' si es que existen Items cuyo array de 'Price Per Unit' tenga mas de 1 valor
resultado = item_prices_array[item_prices_array.apply(len)>1]
print(f'Existen {len(resultado)} items que poseen mas de 1 Price Per Unit')

Existen 0 items que poseen mas de 1 Price Per Unit


Por lo tanto podemos garantizar que cada Item tiene un solo 'Price Per Unit'.
Con esta información podriamos restaurar algunos items nulos si logramos identificar los items mediante el precio unitario y categoria **(siempre y cuando no exista mas de un item en una misma categoria que tenga el mismo precio unitario)**

## Limpiando nulos en Descuentos Aplicados

In [47]:
# Buscamos los valores nulos en Descuento Aplicado
filtro_null_descuento = df.loc[
      df['Discount Applied'].isnull(),
      ['Discount Applied']
    ]
filtro_null_descuento.shape

(0, 1)

In [48]:
# Aqui ya no vemos forma de recuperar los nulls en la columna de descuento aplicado
df.drop(filtro_null_descuento.index, axis=0, inplace=True)

df.isna().sum()

,0
Transaction ID,0
Customer ID,0
Category,0
Item,0
Price Per Unit,0
Quantity,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


## Transformacion Manual

In [49]:
# Imports para la transformacion manual y automatica para IA
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [50]:

df_manual_prueba = df

df_manual = df_manual_prueba.copy()

# Label Encoding (Palabras a Números secuenciales)
# Nota para Lector : LabelEncoder toma la cantidad de variables y les asigna un numero de 0 a N
# Esto se interpreta por los modelos como que el numero mayor es mas importante que el menor
# Como si la variable fuese una Cualitativa Ordinal, es por esta misma razon que en lo personal
# Diria que lo correcto es transformar las cualitativas a 0 y 1 en un pipeline
# Ademas , no considero que para un modelo predictivo se requieran datos como IDs
# Pero para la evaluacion , dejare el codigio igualmente
le = LabelEncoder()
# IDs
# 'Category', 'Item', 'Payment Method', 'Location', 'Discount Applied'
df_manual['Transaccion_ID'] = le.fit_transform(df_manual['Transaction ID'])
df_manual['Cliente_ID'] = le.fit_transform(df_manual['Customer ID'])

# Datos rrelevantes
df_manual['Categoria'] = le.fit_transform(df_manual['Category'])
df_manual['Item'] = le.fit_transform(df_manual['Item'])
df_manual['Metodo_de_pago'] = le.fit_transform(df_manual['Payment Method'])
df_manual['Locacion'] = le.fit_transform(df_manual['Location'])

# Esta si la consideramos correcta
df_manual['Descuento_Aplicado'] = le.fit_transform(df_manual['Discount Applied'])

# Min-Max Scaling (Escalar entre 0 y 1)
# 'Price Per Unit', 'Quantity', 'Total Spent'
mm_scaler = MinMaxScaler()
df_manual['Precio_Unitario_Norm'] = mm_scaler.fit_transform(df_manual[['Price Per Unit']])
df_manual['Cantidad_Norm'] = mm_scaler.fit_transform(df_manual[['Quantity']])
df_manual['Total_Norm'] = mm_scaler.fit_transform(df_manual[['Total Spent']])

df_manual.head(5)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied,Transaccion_ID,Cliente_ID,Categoria,Metodo_de_pago,Locacion,Descuento_Aplicado,Precio_Unitario_Norm,Cantidad_Norm,Total_Norm
0,TXN_6867343,CUST_09,Patisserie,7,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,5159,8,7,2,1,1,0.375000,1.000000,0.444444
1,TXN_3731986,CUST_22,Milk Products,62,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,2398,21,6,2,1,1,0.666667,0.888889,0.632099
2,TXN_9303719,CUST_02,Butchers,17,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,7363,1,1,1,1,0,0.458333,0.111111,0.093827
4,TXN_4575373,CUST_05,Food,172,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,3169,4,4,2,1,0,0.208333,0.666667,0.203704
6,TXN_3652209,CUST_07,Food,84,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,True,2341,6,4,1,0,1,0.000000,0.777778,0.086420


## Transformacion con Pipeline

In [51]:
# Aqui es mejor usar OneHotEncoder ya que las variables no ordinales, se marcaran como 1 y sus demas variantes como 0
# Usamos cat para Variables Categoricas y num para Variables Numericas
# Ignoramos los IDs porque no los consideramos como un dato importante para el analisis


df_auto = df_manual_prueba.copy()

pipeline = ColumnTransformer(transformers=[
('cat', OneHotEncoder(), ['Category', 'Item', 'Payment Method', 'Location', 'Discount Applied']),
('num', StandardScaler(), ["Price Per Unit",	"Quantity",	"Total Spent"])
])
X_transf = pipeline.fit_transform(df_auto)
print(X_transf)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 63864 stored elements and shape (7983, 218)>
  Coords	Values
  (0, 7)	1.0
  (0, 15)	1.0
  (0, 210)	1.0
  (0, 212)	1.0
  (0, 214)	1.0
  (0, 215)	-0.45817841409096893
  (0, 216)	1.5601321454583592
  (0, 217)	0.5799809322084776
  (1, 6)	1.0
  (1, 70)	1.0
  (1, 210)	1.0
  (1, 212)	1.0
  (1, 214)	1.0
  (1, 215)	0.522722204935061
  (1, 216)	1.208983647736636
  (1, 217)	1.3846870424575786
  (2, 1)	1.0
  (2, 25)	1.0
  (2, 209)	1.0
  (2, 212)	1.0
  (2, 213)	1.0
  (2, 215)	-0.17792109436924608
  (2, 216)	-1.2490558363154272
  (2, 217)	-0.923548905362211
  (3, 4)	1.0
  :	:
  (7979, 217)	-0.955313620240465
  (7980, 7)	1.0
  (7980, 95)	1.0
  (7980, 210)	1.0
  (7980, 211)	1.0
  (7980, 213)	1.0
  (7980, 215)	-1.7193363528387218
  (7980, 216)	1.208983647736636
  (7980, 217)	-0.9023724287767083
  (7981, 0)	1.0
  (7981, 144)	1.0
  (7981, 208)	1.0
  (7981, 212)	1.0
  (7981, 213)	1.0
  (7981, 215)	-1.5792076929778602
  (7981, 216)	1.20898364773